In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not set")
else:
    print("GROQ_API_KEY is set")


GROQ_API_KEY is set


## PART 1 — Getting Started with Groq API

**Task 1: Groq API Setup & Basic Chat**
1. Create a Groq account and generate an API key.
2. Install Groq Python SDK.
3. Call a Groq LLM (example: llama3-70b or latest available).
4. Send a simple prompt and print the response.


In [2]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
)


In [3]:
res = llm.invoke("What is the capital of France?")
print(res.content)


The capital of France is **Paris**.



**Task 2: Build a Groq Chatbot (Core Logic)**
1. Create a function:
```
def groq_chat(prompt: str):
    # returns model response
```
2. Handle system + user messages.
3. Test chatbot with multiple queries.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

In [5]:
def groq_chat(prompt: str):
    prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant"),
        ("user", "{prompt}"),
    ])
    chain = prompt_template | llm
    return chain.invoke({"prompt": prompt})


In [6]:
qsns = [
    "what is python in programming?",
    "what is django in web development?",
    "what is machine learning in artificial intelligence?",
    "what is deep learning in artificial intelligence?",
    "what is natural language processing in artificial intelligence?",
    "what is computer vision in artificial intelligence?",
    "what is artificial intelligence in computer science?",   
]

for qsn in qsns:
    print(f"Question: {qsn}")
    print(groq_chat(qsn))
    print("-"*100)

Question: what is python in programming?
content='**Python in a nutshell**\n\nPython is a *high‑level, interpreted, general‑purpose programming language* that was created by Guido van Rossum and first released in 1991. It’s designed to be easy to read and write, which makes it a popular choice for beginners and professionals alike.\n\n| Feature | What it means |\n|---------|---------------|\n| **High‑level** | You write code that looks like natural language (e.g., `print("Hello")`) instead of low‑level machine instructions. |\n| **Interpreted** | The Python interpreter reads and executes your code line‑by‑line, so you can test and run programs instantly without a separate compilation step. |\n| **General‑purpose** | You can use it for web development, data analysis, machine learning, automation, scripting, scientific computing, game development, and more. |\n| **Dynamic typing** | Variables don’t need explicit type declarations; the interpreter figures it out at runtime. |\n| **Readabl


## PART 2 — Groq + RAG (Optional but Recommended)

**Task 3: Groq-Based RAG Pipeline**
1. Retrieve relevant document chunks using a vector store.
2. Construct a RAG prompt using retrieved context.
3. Pass the prompt to Groq LLM.
4. Return grounded answers.

In [7]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

/var/folders/3k/2r4fckl56zxdspgzm1tw5pdw0000gn/T/ipykernel_26981/660606444.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
/Users/abhishekroy/Documents/tutedude/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Large_language_model")

documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512)
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)


In [9]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [10]:
retriever = vectorstore.as_retriever()

system_template = ChatPromptTemplate.from_template(
     """
        You are a helpful assistant that can answer questions about the web page.
        You are given a question and a context.
        You need to answer the question based on the context.
        ---
        Context:
        {context}
        ---
        Question:
        {question}
        ---
    """
)

def format_docs(docs):
    return "\n\n".join([
        f"Document {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(docs)
    ])



In [11]:
chain = {
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
} | system_template | llm | StrOutputParser()

chain.invoke("What is use of LLM?")

'Large Language Models (LLMs) are used for a wide range of tasks that involve understanding, generating, and manipulating natural language. Based on the provided context, some of the key uses include:\n\n| Use | Description |\n|-----|-------------|\n| **Text generation & content creation** | LLMs can produce coherent, fluent text on demand, making them useful for drafting articles, reports, creative writing, and other content‑creation tasks. |\n| **Information retrieval & summarization** | They can answer questions, summarize documents, and retrieve relevant information from large knowledge bases. |\n| **Conversational agents & chatbots** | LLMs power chat interfaces that can hold natural conversations, answer user queries, and provide general assistance. |\n| **Mental‑health support** | Some users employ LLMs to seek therapy‑like conversations or emotional support for issues such as anxiety, depression, or loneliness. (Note: safety concerns exist, including hallucinations and potentia

**Task 4: Prompt Template for Groq RAG**
1. Create a structured prompt template:
   - System instructions
   - Retrieved context
   - User question
2. Ensure answers are based only on context.

**Note**
- system says: answer ONLY from context, else say you dont know
- tested: "What is use of LLM?" → grounded answer
- tested: "capital of France?" → model correctly said context doesnt have it


In [12]:

rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.
Use ONLY the context below to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:
"""
)

def format_docs(docs):
    return "\n\n".join(
        f"Document {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

rag_chain = {
    "context": retriever | format_docs,
    "question": RunnablePassthrough(),
} | rag_prompt | llm | StrOutputParser()


In [13]:
print(rag_chain.invoke("What is use of LLM?"))


LLMs are used primarily to generate text and provide conversational support. In clinical and mental‑health contexts, people are turning to them for therapy‑like or emotional‑support conversations, such as help with anxiety, depression, loneliness, and similar concerns.


In [14]:
# should refuse if not in wiki context
print(rag_chain.invoke("What is the capital of France?"))


I don't know.


## PART 3 — Building API using FastAPI

**Task 5: Create FastAPI Application**
1. Create a FastAPI app (`app.py` in this folder — not main.py, same idea).
2. Add a health check endpoint: `GET /health`
3. Add a POST endpoint for chatbot: `POST /chat`

Request:
```
{ "question": "Your question here" }
```
Response:
```
{ "answer": "model response" }
```

Run from this folder:
```
uv run uvicorn app:app --reload
```
Then open http://127.0.0.1:8000/docs


In [15]:
import requests


In [16]:
url = "http://127.0.0.1:8000"

response = requests.get(f"{url}/health")
print(response.json())


{'status': 'ok'}


**Task 6: FastAPI + Groq Integration**
1. Inside `/chat`, call Groq chatbot logic (`ChatGroq` + prompt chain in `app.py`).
2. Handle errors with try/except → HTTP 500.
3. Request validation via Pydantic `ChatRequest` / `ChatResponse`.

**gotcha:** prompt must include `{question}` or the model never sees the user text.


In [17]:
# Task 6 — chat endpoint (needs uvicorn running)
response = requests.post(
    f"{url}/chat",
    json={"question": "What is the capital of France?"},
)
print(response.status_code)
print(response.json())


200
{'answer': 'The capital of France is **Paris**.'}


In [18]:
# validation check — empty question should 422
bad = requests.post(f"{url}/chat", json={"question": ""})
print("empty question status:", bad.status_code)
print(bad.json())


empty question status: 422
{'detail': [{'type': 'string_too_short', 'loc': ['body', 'question'], 'msg': 'String should have at least 1 character', 'input': '', 'ctx': {'min_length': 1}}]}


## PART 4 — Serving the Groq App

**Task 7: Run & Test FastAPI App Locally**
1. Run: `uv run uvicorn app:app --reload` (from `assignment-26/`)
2. Test with:
   - Browser → http://127.0.0.1:8000/health
   - Swagger UI → http://127.0.0.1:8000/docs
   - notebook `requests` cells / Postman on `POST /chat`

Works for me: `/health` → `{"status":"ok"}`, `/chat` returns Groq answer.


**Task 8 — Production Readiness (Basic)**
1. `requirements.txt` in this folder (fastapi, uvicorn, langchain-groq, …)
2. API keys via `.env` (`GROQ_API_KEY`, and `OPENAI_API_KEY` for embeddings in RAG)
3. basic logging in `app.py` (`logging.info` on each chat request)


In [19]:

from pathlib import Path

print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("requirements.txt exists:", Path("requirements.txt").exists())
print("app.py exists:", Path("app.py").exists())


GROQ_API_KEY set: True
OPENAI_API_KEY set: True
requirements.txt exists: True
app.py exists: True


## PART 5 — Mini Project Validation

**Task 9: End-to-End Demo**
1. Call `/chat` with different questions.
2. Check latency-ish feel, sensible answers, stable JSON responses.


In [20]:
# Task 9 — end-to-end /chat demo
import time

demo_qs = [
    "What is Python?",
    "Explain machine learning in one sentence.",
    "Name three use cases of LLMs.",
    "What is FastAPI used for?",
    "Say hello in one short line.",
]

for q in demo_qs:
    t0 = time.time()
    r = requests.post(f"{url}/chat", json={"question": q})
    dt = time.time() - t0
    print(f"Q: {q}")
    print(f"status={r.status_code} latency={dt:.2f}s")
    print(f"A: {r.json().get('answer', r.text)[:300]}")
    print("-" * 80)


Q: What is Python?
status=200 latency=0.23s
A: Python is a high‑level, general‑purpose programming language known for its readability and simplicity. It is interpreted, dynamically typed, and supports multiple programming paradigms (procedural, object‑oriented, functional). Python’s extensive standard library and vibrant ecosystem of third‑party
--------------------------------------------------------------------------------
Q: Explain machine learning in one sentence.
status=200 latency=0.23s
A: Machine learning is a branch of artificial intelligence that trains algorithms to recognize patterns in data and improve their performance on tasks without being explicitly programmed.
--------------------------------------------------------------------------------
Q: Name three use cases of LLMs.
status=200 latency=0.25s
A: **Three common use cases for large language models (LLMs):**

1. **Content creation & editing** – drafting articles, blogs, marketing copy, or social‑media posts, and prov

**Task 10: Observations & Insights**
Write short answers:

1. Why Groq is suitable for real-time apps → Groq is built for fast inference (LPU). responses feel snappy, so chat / API demos dont feel laggy compared to slower hosted models.
2. Groq vs OpenAI latency (conceptual) → OpenAI is strong quality + ecosystem; Groq usually wins on raw speed for supported open models. pick Groq when low latency matters, OpenAI when you want their specific models / tooling.
3. Benefits of API-first GenAI architecture → wrap the model behind FastAPI (`/health`, `/chat`) so any client (notebook, web, mobile) can call it. easier to swap models, add logging/validation, and deploy without rewriting the UI.
